# TREC Search Engine — Google Colab
**BM25F + Phrase/Proximity + WordNet expansion**

---

## Setup
Upload your TREC dataset to Google Drive:
```
MyDrive/
└── IR_Assignment2/
    └── dataset/       ← any folder name
        ├── TREC-Disk-4/
        └── TREC-Disk-5/
```
All source code is embedded in this notebook — no extra files needed.

**Run cells top to bottom. Once the index is built it persists on Drive — skip Section 3 on future sessions.**

---
## Section 1 — Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys

# ── Edit this path to match your Drive layout ────────────────────────────
DATASET_ROOT = '/content/drive/MyDrive/IR_Assignment2'
# ─────────────────────────────────────────────────────────────────────────

# Index is saved on Drive so it survives session restarts
INDEX_DIR = os.path.join(DATASET_ROOT, 'index_data')
os.makedirs(INDEX_DIR, exist_ok=True)

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

print('Dataset root :', DATASET_ROOT)
print('Index dir    :', INDEX_DIR)

In [ ]:
import nltk
for r in ('stopwords', 'wordnet', 'averaged_perceptron_tagger_eng'):
    nltk.download(r, quiet=True)
print('NLTK ready.')

---
## Section 2 — Write module files
Writes the search-engine source code to `/content/`. Run once per session.

In [ ]:
%%writefile /content/config.py
# config.py — written by notebook (do not edit manually)
import os, sys as _sys

_nb = _sys.modules.get('__main__')
INDEX_DIR    = getattr(_nb, 'INDEX_DIR',    '/content/index_data')
_dataset_root = getattr(_nb, 'DATASET_ROOT', '/content')

INDEX_FILE      = os.path.join(INDEX_DIR, 'inverted_index.db')
DOC_MAP_FILE    = os.path.join(INDEX_DIR, 'doc_map.pkl')
DOC_STATS_FILE  = os.path.join(INDEX_DIR, 'doc_stats.pkl')
COLL_STATS_FILE = os.path.join(INDEX_DIR, 'collection_stats.pkl')
SNIPPETS_FILE   = os.path.join(INDEX_DIR, 'doc_snippets.pkl')

SNIPPET_LENGTH = 200

def _find_disk(base, disk_name):
    for dirpath, dirnames, _ in os.walk(base):
        for d in dirnames:
            if d == disk_name:
                candidate = os.path.join(dirpath, d)
                inner = os.path.join(candidate, disk_name)
                return inner if os.path.isdir(inner) else candidate
    raise FileNotFoundError(
        f"Could not find '{disk_name}' under '{base}'. "
        "Check your Drive layout."
    )

DISK4 = _find_disk(_dataset_root, 'TREC-Disk-4')
DISK5 = _find_disk(_dataset_root, 'TREC-Disk-5')
print(f'[config] TREC-Disk-4 = {DISK4}')
print(f'[config] TREC-Disk-5 = {DISK5}')

COLLECTIONS = [
    (os.path.join(DISK4, 'FT'),       'FT'),
    (os.path.join(DISK4, 'FR94'),     'FR94'),
    (os.path.join(DISK4, 'CR_103RD'), 'CR'),
    (os.path.join(DISK5, 'FBIS'),     'FBIS'),
    (os.path.join(DISK5, 'LATIMES'),  'LATIMES'),
]

DO_STEM             = True
DO_REMOVE_STOPWORDS = True
K1      = 1.2
B_TITLE = 0.75
B_BODY  = 0.75
W_TITLE = 5.0
W_BODY  = 1.0
PHRASE_BONUS        = 1.5
PROXIMITY_WINDOW    = 8
PROXIMITY_BONUS_MAX = 0.5
EXPANSION_GAMMA         = 0.3
MAX_EXPANSIONS_PER_TERM = 3
MAX_DF_RATIO            = 0.10
MIN_COOCCURRENCE        = 1
MAX_POSITIONS_PER_FIELD = 20
SPIMI_CHUNK_SIZE        = 20_000
MAX_BODY_CHARS          = 200_000


In [ ]:
%%writefile /content/preprocess.py
"""
preprocess.py — Text normalisation pipeline (Stage A1 / B1).

The SAME pipeline is applied to both documents and queries so that
stemmed query terms match stemmed index terms correctly.

Pipeline:
  1. Lowercase
  2. Strip HTML/SGML markup & PJG processing instructions
  3. Tokenise with fast regex (replaces slow NLTK word_tokenize)
  4. Remove stopwords
  5. Porter stemming (via NLTK PorterStemmer)

Returns: list of (surface_token, normalised_token, position)
so callers can track both the stemmed form used in the index and the
original position (needed for phrase/proximity matching).
"""

import re
from functools import lru_cache
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import config

# ---------------------------------------------------------------------------
# One-time NLTK resource downloads (only needed for stopwords/wordnet/tagger)
# word_tokenize is no longer used so punkt is not required
# ---------------------------------------------------------------------------
for _resource in ("stopwords", "wordnet", "averaged_perceptron_tagger_eng"):
    try:
        nltk.data.find(f"corpora/{_resource}")
    except LookupError:
        try:
            nltk.download(_resource, quiet=True)
        except Exception:
            pass

# ---------------------------------------------------------------------------
# Module-level singletons (initialised once)
# ---------------------------------------------------------------------------
_stemmer   = PorterStemmer()
_stopwords = set(stopwords.words("english"))

# Cache stemming results — the vocabulary is ~300K unique words but we process
# 100M+ tokens, so each word only gets stemmed once instead of thousands of times.
@lru_cache(maxsize=None)
def _stem(word: str) -> str:
    return _stemmer.stem(word)

# Regex to strip XML/SGML tags and HTML comments
_RE_TAG    = re.compile(r"<!--.*?-->|<[^>]+>", re.DOTALL)
_RE_SPACE  = re.compile(r"\s+")

# Fast tokeniser: extract runs of ASCII letters only.
# Handles hyphenated words by splitting on hyphens (e.g. "well-known" -> ["well","known"])
# This replaces NLTK word_tokenize and is ~8x faster.
_RE_WORDS  = re.compile(r"[a-z]+")


def _strip_markup(text: str) -> str:
    """Remove all XML/SGML tags and HTML comments from *text*."""
    text = _RE_TAG.sub(" ", text)
    text = text.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">")
    text = text.replace("&blank;", " ").replace("&nbsp;", " ")
    return _RE_SPACE.sub(" ", text).strip()


def normalise(text: str) -> list[tuple[str, str, int]]:
    """
    Normalise *text* and return a list of (surface, stemmed, position) triples.

    *surface*  — token as it appears after lowercasing (used for display)
    *stemmed*  — term stored / looked up in the index
    *position* — 0-based word offset in the token stream (used for phrase/proximity)
    """
    text = _strip_markup(text).lower()

    result: list[tuple[str, str, int]] = []
    pos = 0

    for tok in _RE_WORDS.findall(text):
        if len(tok) < 2:          # skip single-letter tokens
            pos += 1
            continue

        if config.DO_REMOVE_STOPWORDS and tok in _stopwords:
            pos += 1
            continue

        stemmed = _stem(tok) if config.DO_STEM else tok
        result.append((tok, stemmed, pos))
        pos += 1

    return result


def terms(text: str) -> list[str]:
    """Convenience wrapper: return just the stemmed terms (no positions)."""
    return [stemmed for _, stemmed, _ in normalise(text)]


def terms_with_positions(text: str) -> list[tuple[str, int]]:
    """Return (stemmed_term, position) pairs."""
    return [(stemmed, p) for _, stemmed, p in normalise(text)]


In [ ]:
%%writefile /content/parse_docs.py
"""
parse_docs.py — Regex-based SGML document parser for all 5 TREC collections.

Each collection has its own tag conventions; we handle them with targeted
regexes rather than a strict XML parser (the SGML is not valid XML).

Public API
----------
iter_collection(root_dir, collection_type) -> Iterator[dict]
    Yields dicts with keys: 'docno', 'title', 'body'

iter_all_collections(collections) -> Iterator[dict]
    Convenience wrapper over config.COLLECTIONS.
"""

import os
import re
from typing import Iterator

# ---------------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------------

def _tag(name: str) -> re.Pattern:
    """Compile a pattern that extracts the content of a single SGML tag."""
    return re.compile(
        rf"<{name}[^>]*>(.*?)</{name}>",
        re.DOTALL | re.IGNORECASE,
    )

def _extract(pattern: re.Pattern, text: str, default: str = "") -> str:
    m = pattern.search(text)
    return m.group(1).strip() if m else default

def _strip_inner_tags(text: str) -> str:
    """Remove inner XML/SGML tags, keeping their text content."""
    return re.sub(r"<[^>]+>", " ", text)

def _strip_pjg(text: str) -> str:
    """Remove PJG processing instructions used in FR94."""
    # <!-- PJG ... --> style comments
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    return text

# Compiled patterns for each field
_P_DOCNO    = _tag("DOCNO")
_P_TEXT     = _tag("TEXT")
_P_HEADLINE = _tag("HEADLINE")
_P_TI       = _tag("TI")
_P_H3       = _tag("H3")
_P_TTL      = _tag("TTL")       # Congressional Record title (inside TEXT)
_P_SO       = _tag("SO")        # SO section at end of CR TEXT (speaker info)

# DOC splitter — one file may hold many <DOC>…</DOC> records
_P_DOC = re.compile(r"<DOC>(.*?)</DOC>", re.DOTALL | re.IGNORECASE)


def _split_docs(file_text: str) -> list[str]:
    return _P_DOC.findall(file_text)


# ---------------------------------------------------------------------------
# Per-collection parsers
# ---------------------------------------------------------------------------

def _parse_ft(raw: str) -> dict | None:
    """Financial Times: HEADLINE → title, TEXT → body."""
    docno = _extract(_P_DOCNO, raw).strip()
    if not docno:
        return None
    title = _strip_inner_tags(_extract(_P_HEADLINE, raw))
    body  = _strip_inner_tags(_extract(_P_TEXT, raw))
    return {"docno": docno, "title": title, "body": body}


def _parse_fbis(raw: str) -> dict | None:
    """FBIS: TI inside H3 → title, TEXT → body."""
    docno = _extract(_P_DOCNO, raw).strip()
    if not docno:
        return None
    # Title may be inside <H3><TI>…</TI></H3> or standalone <TI>…</TI>
    h3_block = _extract(_P_H3, raw)
    if h3_block:
        title = _strip_inner_tags(_extract(_P_TI, h3_block) or h3_block)
    else:
        title = _strip_inner_tags(_extract(_P_TI, raw))
    body = _strip_inner_tags(_extract(_P_TEXT, raw))
    return {"docno": docno, "title": title, "body": body}


def _parse_fr94(raw: str) -> dict | None:
    """
    Federal Register 1994: heavily markup-laden TEXT, no dedicated title tag.
    We strip PJG instructions and use an empty title (FR94 docs are body-only
    for ranking purposes).
    """
    docno = _extract(_P_DOCNO, raw).strip()
    if not docno:
        return None
    text_raw = _extract(_P_TEXT, raw)
    body = _strip_inner_tags(_strip_pjg(text_raw))
    return {"docno": docno, "title": "", "body": body}


def _parse_cr(raw: str) -> dict | None:
    """
    Congressional Record: <TTL>…</TTL> sits *inside* the <TEXT> block.
    Title = TTL content; body = TEXT minus TTL and SO sections.
    """
    docno = _extract(_P_DOCNO, raw).strip()
    if not docno:
        return None
    text_raw = _extract(_P_TEXT, raw)
    # Extract title from TTL (may be several; use first)
    title = _strip_inner_tags(_extract(_P_TTL, text_raw))
    # Remove TTL and SO blocks from body
    body_raw = re.sub(r"<TTL>.*?</TTL>", " ", text_raw, flags=re.DOTALL | re.IGNORECASE)
    body_raw = re.sub(r"<SO>.*?</SO>",   " ", body_raw,  flags=re.DOTALL | re.IGNORECASE)
    body = _strip_inner_tags(body_raw)
    return {"docno": docno, "title": title, "body": body}


def _parse_latimes(raw: str) -> dict | None:
    """LA Times: HEADLINE → title (contains <P> tags), TEXT → body."""
    docno = _extract(_P_DOCNO, raw).strip()
    if not docno:
        return None
    title = _strip_inner_tags(_extract(_P_HEADLINE, raw))
    body  = _strip_inner_tags(_extract(_P_TEXT, raw))
    return {"docno": docno, "title": title, "body": body}


# Map collection type tag → parser function
_PARSERS = {
    "FT":      _parse_ft,
    "FBIS":    _parse_fbis,
    "FR94":    _parse_fr94,
    "CR":      _parse_cr,
    "LATIMES": _parse_latimes,
}

# ---------------------------------------------------------------------------
# File / directory walkers
# ---------------------------------------------------------------------------

_SKIP_NAMES = {".DS_Store", "_.DS_Store", "MD5SUM", "READMEFT", "READMEFR",
               "READCHG", "READFRCG", "CREDTD", "CRHDTD", "FR94DTD",
               "FTDTD", "FBIS.DTD", "LA.DTD"}
_SKIP_EXTS  = {".sgml", ".dtd", ".xml"}


def _should_skip(name: str) -> bool:
    if name.startswith("._"):
        return True
    if name in _SKIP_NAMES:
        return True
    ext = os.path.splitext(name)[1].lower()
    if ext in _SKIP_EXTS:
        return True
    return False


def _read_file(path: str) -> str:
    """Read a TREC data file with Latin-1 fallback (some files are not UTF-8)."""
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            with open(path, "r", encoding=enc, errors="replace") as fh:
                return fh.read()
        except Exception:
            continue
    return ""


def iter_collection(root_dir: str, collection_type: str) -> Iterator[dict]:
    """
    Walk *root_dir* recursively, parse every document file, and yield
    dicts with keys 'docno', 'title', 'body'.
    """
    parser = _PARSERS.get(collection_type)
    if parser is None:
        raise ValueError(f"Unknown collection type: {collection_type!r}")

    for dirpath, dirnames, filenames in os.walk(root_dir):
        # Skip DTD subdirectories
        dirnames[:] = [d for d in dirnames if d.upper() != "DTDS"]
        for fname in filenames:
            if _should_skip(fname):
                continue
            fpath = os.path.join(dirpath, fname)
            text  = _read_file(fpath)
            if not text:
                continue
            for raw in _split_docs(text):
                doc = parser(raw)
                if doc and doc["docno"]:
                    yield doc


def iter_all_collections(collections: list[tuple[str, str]]) -> Iterator[dict]:
    """Iterate over all (root_dir, collection_type) pairs from config.COLLECTIONS."""
    for root_dir, ctype in collections:
        if not os.path.isdir(root_dir):
            print(f"[WARN] Collection directory not found: {root_dir}")
            continue
        yield from iter_collection(root_dir, ctype)


In [ ]:
%%writefile /content/index_store.py
"""
index_store.py — Dict-like wrapper around the SQLite inverted index.

Allows rank.py and query_expand.py to use .get(), `in`, and [] on the
on-disk index without loading the whole thing into RAM.
"""

import pickle
import sqlite3


class SQLiteIndex:
    """Read-only dict-like interface to the SQLite inverted index."""

    def __init__(self, path: str):
        self._conn = sqlite3.connect(path, check_same_thread=False)

    def get(self, term: str, default=None):
        row = self._conn.execute(
            "SELECT df, postings FROM idx WHERE term=?", (term,)
        ).fetchone()
        if row is None:
            return default
        return (row[0], pickle.loads(row[1]))

    def __contains__(self, term: str) -> bool:
        return self._conn.execute(
            "SELECT 1 FROM idx WHERE term=?", (term,)
        ).fetchone() is not None

    def __getitem__(self, term: str):
        result = self.get(term)
        if result is None:
            raise KeyError(term)
        return result

    def __len__(self) -> int:
        return self._conn.execute("SELECT COUNT(*) FROM idx").fetchone()[0]

    def close(self):
        self._conn.close()


In [ ]:
%%writefile /content/build_index.py
"""
build_index.py — Stage A (offline): build and persist the inverted index.

Run once before searching:
    python build_index.py

Algorithm: SPIMI (Single-Pass In-Memory Indexing)
--------------------------------------------------
Rather than accumulating all postings in RAM at once (which OOMs on the full
TREC Disk 4/5 corpus), we:
  1. Process SPIMI_CHUNK_SIZE documents at a time into a small in-memory dict.
  2. Flush each chunk to a compact partial-index pickle file on disk.
  3. After all documents are processed, sequentially merge the partial files
     into the final inverted index and remove the temporary run files.

On-disk layout (index_data/)
----------------------------
inverted_index.pkl   — dict[term, (df, postings_list)]
doc_map.pkl          — list[docno_string]  (index = integer doc_id)
doc_stats.pkl        — list[(title_len, body_len)]
collection_stats.pkl — dict with N, avg_title_len, avg_body_len
doc_snippets.pkl     — list[str]  short preview per document
"""

import os
import sys
import pickle
import shutil
import time
import itertools
import multiprocessing
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor

import config
import preprocess
import parse_docs

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _save(obj, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as fh:
        pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  saved: {path}  ({os.path.getsize(path) / 1e6:.1f} MB)")


def _save_checkpoint(runs_dir: str, doc_map: list, doc_stats: list,
                     doc_snippets: list, run_paths: list) -> None:
    path = os.path.join(runs_dir, "checkpoint.pkl")
    with open(path, "wb") as fh:
        pickle.dump({
            "doc_map":      doc_map,
            "doc_stats":    doc_stats,
            "doc_snippets": doc_snippets,
            "run_paths":    run_paths,
        }, fh, protocol=pickle.HIGHEST_PROTOCOL)


def _load_checkpoint(runs_dir: str) -> dict | None:
    path = os.path.join(runs_dir, "checkpoint.pkl")
    if not os.path.exists(path):
        return None
    with open(path, "rb") as fh:
        ckpt = pickle.load(fh)
    # Verify all run files referenced by the checkpoint still exist
    if not all(os.path.exists(p) for p in ckpt["run_paths"]):
        print("  WARNING: checkpoint references missing run files — starting fresh.")
        return None
    return ckpt


def _flush_run(build_idx: dict, runs_dir: str, run_num: int) -> str:
    """
    Convert *build_idx* (nested dict) to compact postings tuples and write
    a partial-index pickle.  Returns the path of the written file.
    """
    partial: dict[str, tuple] = {}
    for term, doc_dict in build_idx.items():
        postings = []
        for doc_id, fields in sorted(doc_dict.items()):
            t_pos = tuple(sorted(fields["t"]))
            b_pos = tuple(sorted(fields["b"]))
            postings.append((doc_id, len(t_pos), len(b_pos), t_pos, b_pos))
        partial[term] = (len(postings), postings)

    path = os.path.join(runs_dir, f"run_{run_num:04d}.pkl")
    with open(path, "wb") as fh:
        pickle.dump(partial, fh, protocol=pickle.HIGHEST_PROTOCOL)
    return path


_MERGE_BATCH = 10_000   # terms fetched/written per SQLite transaction


def _merge_runs(run_paths: list[str], output_path: str) -> int:
    """
    Merge all partial-index files into a SQLite database on disk.

    Peak RAM ≈ one MERGE_BATCH slice of one run at a time.
    Returns the number of unique terms written.
    """
    import sqlite3 as _sql

    conn = _sql.connect(output_path)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=OFF")
    conn.execute("PRAGMA cache_size=-262144")   # 256 MB page cache
    conn.execute(
        "CREATE TABLE idx (term TEXT PRIMARY KEY, df INTEGER, postings BLOB)"
    )

    term_count = 0

    for i, path in enumerate(run_paths):
        print(f"  merging run {i + 1}/{len(run_paths)} …", flush=True)
        with open(path, "rb") as fh:
            partial: dict[str, tuple] = pickle.load(fh)

        items = list(partial.items())
        del partial

        for start in range(0, len(items), _MERGE_BATCH):
            batch = items[start : start + _MERGE_BATCH]
            terms = [t for t, _ in batch]

            # Fetch any already-existing rows for this batch in one query
            placeholders = ",".join("?" * len(terms))
            existing = {
                row[0]: (row[1], pickle.loads(row[2]))
                for row in conn.execute(
                    f"SELECT term, df, postings FROM idx WHERE term IN ({placeholders})",
                    terms,
                )
            }

            to_insert: list[tuple] = []
            to_update: list[tuple] = []
            for term, (df, postings) in batch:
                if term in existing:
                    old_df, old_posts = existing[term]
                    to_update.append((
                        old_df + df,
                        pickle.dumps(old_posts + list(postings),
                                     protocol=pickle.HIGHEST_PROTOCOL),
                        term,
                    ))
                else:
                    to_insert.append((
                        term, df,
                        pickle.dumps(list(postings),
                                     protocol=pickle.HIGHEST_PROTOCOL),
                    ))
                    term_count += 1

            with conn:   # auto-commit this transaction
                if to_insert:
                    conn.executemany("INSERT INTO idx VALUES (?,?,?)", to_insert)
                if to_update:
                    conn.executemany(
                        "UPDATE idx SET df=?, postings=? WHERE term=?", to_update
                    )

    conn.close()
    return term_count


# ---------------------------------------------------------------------------
# Parallel helpers (must be module-level for pickling on Windows)
# ---------------------------------------------------------------------------

def _collect_files(collections: list) -> list[tuple[str, str]]:
    """Return a sorted list of (fpath, ctype) for every data file."""
    files = []
    for root_dir, ctype in collections:
        if not os.path.isdir(root_dir):
            print(f"[WARN] Collection directory not found: {root_dir}")
            continue
        for dirpath, dirnames, filenames in os.walk(root_dir):
            dirnames[:] = sorted(d for d in dirnames if d.upper() != "DTDS")
            for fname in sorted(filenames):
                if not parse_docs._should_skip(fname):
                    files.append((os.path.join(dirpath, fname), ctype))
    return files


def _process_file(args: tuple) -> list:
    """
    Parse and preprocess one data file.  Runs in a worker process.
    Returns a list of (docno, title_tokens, body_tokens, snippet).
    Positions are pre-capped per term to reduce IPC transfer size.
    """
    fpath, ctype = args
    parser = parse_docs._PARSERS.get(ctype)
    if parser is None:
        return []
    text = parse_docs._read_file(fpath)
    if not text:
        return []
    cap = config.MAX_POSITIONS_PER_FIELD
    out = []
    for raw in parse_docs._split_docs(text):
        doc = parser(raw)
        if not doc or not doc["docno"]:
            continue
        title_tok = preprocess.terms_with_positions(doc["title"])
        body_tok  = preprocess.terms_with_positions(doc["body"][:config.MAX_BODY_CHARS])
        # Cap per-term positions here to reduce data sent back to main process
        def _cap(tokens):
            counts: dict = {}
            result = []
            for term, pos in tokens:
                c = counts.get(term, 0)
                if c < cap:
                    result.append((term, pos))
                    counts[term] = c + 1
            return result
        snippet = doc["body"].strip().replace("\n", " ")[:config.SNIPPET_LENGTH]
        out.append((doc["docno"], _cap(title_tok), _cap(body_tok), snippet))
    return out


# ---------------------------------------------------------------------------
# Core indexing logic
# ---------------------------------------------------------------------------

def build() -> None:
    print("=" * 60)
    print("Stage A — Building inverted index  (SPIMI)")
    print(f"  chunk size : {config.SPIMI_CHUNK_SIZE:,} docs")
    print(f"  max pos/field: {config.MAX_POSITIONS_PER_FIELD}")
    print("=" * 60)

    runs_dir = os.path.join(config.INDEX_DIR, "runs")
    os.makedirs(runs_dir, exist_ok=True)

    # ------------------------------------------------------------------
    # Resume from checkpoint if one exists
    # ------------------------------------------------------------------
    ckpt = _load_checkpoint(runs_dir)
    if ckpt:
        doc_map      = ckpt["doc_map"]
        doc_stats    = ckpt["doc_stats"]
        doc_snippets = ckpt["doc_snippets"]
        run_paths    = ckpt["run_paths"]
        docs_to_skip = len(doc_map)
        print(f"  Resuming from checkpoint: {docs_to_skip:,} docs already processed, "
              f"{len(run_paths)} run(s) on disk.")
    else:
        doc_map      = []
        doc_stats    = []
        doc_snippets = []
        run_paths    = []
        docs_to_skip = 0

    # ------------------------------------------------------------------
    # A1 + A2: SPIMI build pass
    # ------------------------------------------------------------------
    # build_idx[term][doc_id] = {"t": [pos, ...], "b": [pos, ...]}
    build_idx: dict = defaultdict(lambda: defaultdict(lambda: {"t": [], "b": []}))

    t0 = time.time()
    doc_count = len(doc_map)

    def _ingest(docno: str, title_tok: list, body_tok: list, snippet: str) -> None:
        """Add one pre-processed document into the in-memory index structures."""
        nonlocal doc_count, build_idx
        doc_id = len(doc_map)
        doc_map.append(docno)
        doc_stats.append((len(title_tok), len(body_tok)))
        doc_snippets.append(snippet)
        for term, pos in title_tok:
            build_idx[term][doc_id]["t"].append(pos)
        for term, pos in body_tok:
            build_idx[term][doc_id]["b"].append(pos)
        doc_count += 1
        if doc_count % config.SPIMI_CHUNK_SIZE == 0:
            run_num = len(run_paths)
            print(
                f"  [{time.time() - t0:5.0f}s]  {doc_count:>7,} docs — "
                f"flushing run {run_num} ({len(build_idx):,} terms) …",
                flush=True,
            )
            path = _flush_run(build_idx, runs_dir, run_num)
            run_paths.append(path)
            _save_checkpoint(runs_dir, doc_map, doc_stats, doc_snippets, run_paths)
            build_idx.clear()
            build_idx = defaultdict(lambda: defaultdict(lambda: {"t": [], "b": []}))

    if docs_to_skip:
        # ----- Sequential resume path -----
        # Preprocessing is the bottleneck, but for a resume we need to skip
        # N docs from the start of the stream so we iterate sequentially.
        print("  (sequential mode for resume)", flush=True)
        all_docs = itertools.islice(
            parse_docs.iter_all_collections(config.COLLECTIONS), docs_to_skip, None
        )
        for doc in all_docs:
            title_tok = preprocess.terms_with_positions(doc["title"])
            body_tok  = preprocess.terms_with_positions(doc["body"][:config.MAX_BODY_CHARS])
            snippet   = doc["body"].strip().replace("\n", " ")[:config.SNIPPET_LENGTH]
            _ingest(doc["docno"], title_tok, body_tok, snippet)
    else:
        # ----- Parallel path for fresh builds -----
        # Each worker parses + preprocesses one file; main process builds index.
        n_workers = max(1, multiprocessing.cpu_count() - 1)
        all_files = _collect_files(config.COLLECTIONS)
        print(f"  Workers: {n_workers}  |  Files: {len(all_files):,}", flush=True)
        with ProcessPoolExecutor(max_workers=n_workers) as executor:
            for file_docs in executor.map(_process_file, all_files, chunksize=8):
                for docno, title_tok, body_tok, snippet in file_docs:
                    _ingest(docno, title_tok, body_tok, snippet)

    # flush any remaining documents
    if build_idx:
        run_num = len(run_paths)
        print(
            f"  [{time.time() - t0:5.0f}s]  {doc_count:>7,} docs — "
            f"flushing final run {run_num} ({len(build_idx):,} terms) …",
            flush=True,
        )
        path = _flush_run(build_idx, runs_dir, run_num)
        run_paths.append(path)
        _save_checkpoint(runs_dir, doc_map, doc_stats, doc_snippets, run_paths)
        del build_idx

    print(f"\n  Total documents : {doc_count:,}")
    print(f"  Runs written    : {len(run_paths)}")
    print(f"  Elapsed so far  : {time.time() - t0:.1f}s")

    # ------------------------------------------------------------------
    # Merge partial runs
    # ------------------------------------------------------------------
    print("\nMerging partial runs …", flush=True)
    t1 = time.time()
    term_count = _merge_runs(run_paths, config.INDEX_FILE)
    print(f"  Unique terms    : {term_count:,}")
    print(f"  Merge time      : {time.time() - t1:.1f}s")

    # Remove temporary run files
    shutil.rmtree(runs_dir, ignore_errors=True)

    # ------------------------------------------------------------------
    # A3: Collection-level statistics
    # ------------------------------------------------------------------
    N = doc_count
    avg_title_len = sum(s[0] for s in doc_stats) / N if N else 0.0
    avg_body_len  = sum(s[1] for s in doc_stats) / N if N else 0.0

    collection_stats = {
        "N":             N,
        "avg_title_len": avg_title_len,
        "avg_body_len":  avg_body_len,
    }
    print(f"\n  avg title len = {avg_title_len:.1f}")
    print(f"  avg body  len = {avg_body_len:.1f}")

    # ------------------------------------------------------------------
    # Persist everything
    # ------------------------------------------------------------------
    print("\nSaving index files …")
    _save(doc_map,          config.DOC_MAP_FILE)
    _save(doc_stats,        config.DOC_STATS_FILE)
    _save(collection_stats, config.COLL_STATS_FILE)
    _save(doc_snippets,     config.SNIPPETS_FILE)

    print(f"\nDone. Total time: {time.time() - t0:.1f}s")


if __name__ == "__main__":
    build()


In [ ]:
%%writefile /content/query_expand.py
"""
query_expand.py — Stage B2: Thesaurus-based query expansion via WordNet.

Design (drift-controlled expansion)
------------------------------------
B2.1  Select expandable terms     → nouns only, high-IDF only
B2.2  Sense handling              → WSD-lite via gloss/example overlap
B2.3  Generate candidates         → synonyms (same synset) only
B2.4  Filter candidates           → stopword, DF, co-occurrence, cap M
B2.5  Weight expansions           → γ < 1 relative to original terms

Public API
----------
expand_query(query_terms, inverted_index, collection_stats, pos_tags=None)
    -> dict[term: str, weight: float]

The returned dict includes original terms (weight=1.0) and expansions
(weight=EXPANSION_GAMMA), ready to be used by the ranker.
"""

import math
import re
from collections import defaultdict

import nltk
from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords as nltk_sw

import config
import preprocess

# Ensure required NLTK data is available
for _r in ("wordnet", "stopwords", "averaged_perceptron_tagger",
           "averaged_perceptron_tagger_eng"):
    try:
        nltk.data.find(f"corpora/{_r}")
    except LookupError:
        try:
            nltk.download(_r, quiet=True)
        except Exception:
            pass

_STOPWORDS = set(nltk_sw.words("english"))
_RE_ALPHA  = re.compile(r"^[a-z]+$")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _idf(term: str, inverted_index: dict, N: int) -> float:
    """Robertson-Sparck Jones IDF (with smoothing)."""
    entry = inverted_index.get(term)
    df = entry[0] if entry else 0
    return math.log((N - df + 0.5) / (df + 0.5) + 1)


def _pos_tag_query(surface_tokens: list[str]) -> list[tuple[str, str]]:
    """POS-tag the original surface tokens and return (token, coarse_pos) pairs."""
    try:
        tagged = nltk.pos_tag(surface_tokens)
        # Coarsen Penn Treebank tags: NN* → 'n', VB* → 'v', JJ* → 'j', RB* → 'r'
        coarse = []
        for tok, tag in tagged:
            if tag.startswith("NN"):
                coarse.append((tok, "n"))
            elif tag.startswith("VB"):
                coarse.append((tok, "v"))
            elif tag.startswith("JJ"):
                coarse.append((tok, "j"))
            elif tag.startswith("RB"):
                coarse.append((tok, "r"))
            else:
                coarse.append((tok, "other"))
        return coarse
    except Exception:
        return [(t, "other") for t in surface_tokens]


def _wsd_lite(surface_token: str, wn_pos: str, query_surface_tokens: list[str]
              ) -> wn.Synset | None:
    """
    Lightweight Word Sense Disambiguation (B2.2).

    Choose the synset whose definition + examples share the most words
    with the rest of the query (context bag-of-words).  Falls back to
    the most-frequent sense (synsets()[0]) if no overlap is found.
    """
    synsets = wn.synsets(surface_token, pos=wn_pos)
    if not synsets:
        return None

    # Build a context set from the other query tokens
    context = set(query_surface_tokens) - {surface_token} - _STOPWORDS

    if not context:
        return synsets[0]   # MFS fallback

    best_synset = synsets[0]
    best_overlap = -1

    for ss in synsets:
        gloss_words = set(re.findall(r"[a-z]+", ss.definition().lower()))
        for ex in ss.examples():
            gloss_words |= set(re.findall(r"[a-z]+", ex.lower()))
        overlap = len(gloss_words & context)
        if overlap > best_overlap:
            best_overlap = overlap
            best_synset  = ss

    return best_synset


def _cooccurrence_ok(candidate: str, original_terms: list[str],
                     inverted_index: dict) -> bool:
    """
    B2.4 co-occurrence filter.

    Return True if *candidate* co-occurs (shares documents) with at least
    one original query term in at least MIN_COOCCURRENCE documents.
    We approximate this by checking posting-list intersection size ≥ 1.
    """
    cand_entry = inverted_index.get(candidate)
    if cand_entry is None:
        return False
    cand_doc_ids = {p[0] for p in cand_entry[1]}

    for orig in original_terms:
        orig_entry = inverted_index.get(orig)
        if orig_entry is None:
            continue
        orig_doc_ids = {p[0] for p in orig_entry[1]}
        if len(cand_doc_ids & orig_doc_ids) >= config.MIN_COOCCURRENCE:
            return True
    return False


# ---------------------------------------------------------------------------
# Map NLTK coarse POS to WordNet POS constant
# ---------------------------------------------------------------------------
_POS_MAP = {"n": wn.NOUN, "v": wn.VERB, "j": wn.ADJ, "r": wn.ADV}


# ---------------------------------------------------------------------------
# Main expansion function
# ---------------------------------------------------------------------------

def expand_query(
    query_terms: list[str],           # stemmed terms (from preprocess)
    surface_tokens: list[str],        # original surface tokens (pre-stem)
    inverted_index: dict,
    collection_stats: dict,
) -> dict[str, float]:
    """
    Expand *query_terms* using WordNet synonyms with seven drift controls.

    Returns a weight dict:
        {term: weight, ...}
    Original terms have weight 1.0; expansions have weight EXPANSION_GAMMA.
    """
    N = collection_stats["N"]

    # Start with original terms (weight 1.0)
    weighted: dict[str, float] = {t: 1.0 for t in query_terms}

    if N == 0:
        return weighted

    # -----------------------------------------------------------------------
    # B2.1  Select expandable terms
    # -----------------------------------------------------------------------
    # POS-tag the surface tokens to identify nouns
    tagged = _pos_tag_query(surface_tokens)

    # Map surface_token → coarse POS for nouns we're willing to expand
    expandable: list[tuple[str, str, str]] = []  # (surface, stemmed, wn_pos)
    for (surface, coarse_pos), stemmed in zip(tagged, query_terms):
        # Only expand nouns (safest for topic preservation)
        if coarse_pos != "n":
            continue
        # Only expand terms that are already in the index (non-zero IDF boost)
        if stemmed not in inverted_index:
            continue
        # Only expand reasonably specific terms (IDF > threshold)
        term_idf = _idf(stemmed, inverted_index, N)
        if term_idf < 1.0:   # skip very generic terms
            continue
        expandable.append((surface, stemmed, wn.NOUN))

    # -----------------------------------------------------------------------
    # B2.2 → B2.4: For each expandable term, pick sense + synonyms + filter
    # -----------------------------------------------------------------------
    for surface, stemmed, wn_pos in expandable:
        # B2.2 WSD-lite sense selection
        synset = _wsd_lite(surface, wn_pos, surface_tokens)
        if synset is None:
            continue

        # B2.3 Collect synonym lemma names (synonyms only — no hypernyms/hyponyms)
        candidates: list[str] = []
        for lemma in synset.lemmas():
            cand = lemma.name().lower().replace("_", " ").replace("-", " ")
            # Keep only single-word candidates
            if " " in cand:
                continue
            if not _RE_ALPHA.match(cand):
                continue
            if cand == surface or cand == stemmed:
                continue
            candidates.append(cand)

        # B2.4 Filter candidates
        accepted: list[tuple[str, float]] = []  # (stemmed_cand, idf)
        for cand_surface in candidates:
            # Stopword / punctuation cleanup
            if cand_surface in _STOPWORDS:
                continue
            if len(cand_surface) < 2:
                continue

            # Stem the candidate so it matches index terms
            cand_stemmed = preprocess.terms(cand_surface)
            if not cand_stemmed:
                continue
            cand_stem = cand_stemmed[0]

            # Don't add if already in the query (even in stemmed form)
            if cand_stem in weighted:
                continue

            # High-DF filter: reject if candidate appears in >MAX_DF_RATIO of docs
            entry = inverted_index.get(cand_stem)
            if entry is None:
                continue
            df_ratio = entry[0] / N
            if df_ratio > config.MAX_DF_RATIO:
                continue

            # Co-occurrence filter
            if not _cooccurrence_ok(cand_stem, query_terms, inverted_index):
                continue

            cand_idf = _idf(cand_stem, inverted_index, N)
            accepted.append((cand_stem, cand_idf))

        # Sort by IDF descending (more specific expansions preferred) and cap
        accepted.sort(key=lambda x: x[1], reverse=True)
        accepted = accepted[: config.MAX_EXPANSIONS_PER_TERM]

        # B2.5 Weight expansions
        # Base weight = EXPANSION_GAMMA; optionally scale by relative IDF
        orig_idf = _idf(stemmed, inverted_index, N)
        for cand_stem, cand_idf in accepted:
            # IDF-relative scaling: expansion IDF / original IDF, clipped to [0.5, 1.0]
            idf_scale = min(1.0, max(0.5, cand_idf / orig_idf)) if orig_idf > 0 else 1.0
            weight = config.EXPANSION_GAMMA * idf_scale
            weighted[cand_stem] = weight

    return weighted


In [ ]:
%%writefile /content/rank.py
"""
rank.py — Scoring functions (BM25F + phrase bonus + proximity bonus).

Scoring formula (Section 7):
    Score(doc, query) = BM25F(doc, original_terms)
                      + γ · BM25F(doc, expanded_terms)
                      + phrase_bonus
                      + proximity_bonus

BM25F (Option 2 — Zaragoza et al. 2004):
    Combine per-field normalised TFs *before* applying saturation, so
    saturation operates on the combined evidence once (avoids double-
    application of the k1 ceiling that Option 1 suffers from).

    normalised_tf_field = tf_field / (1 - b_field + b_field * len_field / avg_len_field)
    combined_tf         = w_title * norm_title_tf + w_body * norm_body_tf
    idf                 = log((N - df + 0.5) / (df + 0.5) + 1)
    bm25f               = idf * combined_tf / (k1 + combined_tf)

Phrase bonus (Metzler & Croft 2005 — term dependence model):
    +PHRASE_BONUS  for each pair of consecutive query terms that appear
    as an exact ordered sequence in either the title or the body.

Proximity bonus:
    For each pair of query terms, +bonus if they appear within
    PROXIMITY_WINDOW tokens of each other (unordered).
    Bonus = PROXIMITY_BONUS_MAX * (1 - gap / PROXIMITY_WINDOW).
"""

import math
from itertools import combinations

import config


# ---------------------------------------------------------------------------
# IDF
# ---------------------------------------------------------------------------

def _idf(df: int, N: int) -> float:
    """Robertson-Sparck Jones IDF with +1 smoothing."""
    return math.log((N - df + 0.5) / (df + 0.5) + 1.0)


# ---------------------------------------------------------------------------
# BM25F score for a single term in a single document
# ---------------------------------------------------------------------------

def _bm25f_term(
    title_tf:    int,
    body_tf:     int,
    title_len:   int,
    body_len:    int,
    avg_title:   float,
    avg_body:    float,
    df:          int,
    N:           int,
    term_weight: float = 1.0,
) -> float:
    """
    Return the BM25F contribution for one (term, document) pair.

    *term_weight* < 1 for expanded terms (γ already applied by the caller).
    """
    if df == 0 or N == 0:
        return 0.0

    # Normalised TF per field (length normalisation)
    norm_title = (title_tf / (1 - config.B_TITLE + config.B_TITLE * title_len / avg_title)
                  if avg_title > 0 else title_tf)
    norm_body  = (body_tf  / (1 - config.B_BODY  + config.B_BODY  * body_len  / avg_body)
                  if avg_body  > 0 else body_tf)

    # Combined TF with field weights
    combined_tf = config.W_TITLE * norm_title + config.W_BODY * norm_body

    idf = _idf(df, N)

    # BM25F saturation (once, on combined TF)
    return term_weight * idf * combined_tf / (config.K1 + combined_tf)


# ---------------------------------------------------------------------------
# Phrase bonus
# ---------------------------------------------------------------------------

def _exact_phrase_positions(positions_a: tuple[int, ...],
                             positions_b: tuple[int, ...]) -> bool:
    """
    Return True if term B immediately follows term A anywhere
    in the position lists (i.e., pos_b == pos_a + 1 for some pair).
    """
    set_a = set(positions_a)
    for pb in positions_b:
        if (pb - 1) in set_a:
            return True
    return False


def _phrase_bonus(query_terms: list[str],
                  posting: tuple) -> float:
    """
    Sum of PHRASE_BONUS for each consecutive query-term pair that forms
    an exact phrase in either the title or the body positions.

    *posting* layout: (doc_id, title_tf, body_tf, title_pos, body_pos)
    """
    if len(query_terms) < 2:
        return 0.0

    _, _, _, title_pos, body_pos = posting
    bonus = 0.0

    for i in range(len(query_terms) - 1):
        # We need position data for both terms in the same doc; however
        # at the point this function is called we only have one posting.
        # The caller passes pre-fetched position tuples directly.
        pass  # handled in score_document

    return bonus


# ---------------------------------------------------------------------------
# Proximity bonus
# ---------------------------------------------------------------------------

def _min_gap(positions_a: tuple[int, ...],
             positions_b: tuple[int, ...]) -> int:
    """
    Find the minimum absolute distance between any position in A and any in B.
    Uses a two-pointer merge since both sequences are sorted.
    """
    if not positions_a or not positions_b:
        return config.PROXIMITY_WINDOW + 1  # signals "no proximity"

    i = j = 0
    min_g = abs(positions_a[0] - positions_b[0])
    while i < len(positions_a) and j < len(positions_b):
        gap = abs(positions_a[i] - positions_b[j])
        if gap < min_g:
            min_g = gap
        if min_g == 1:      # can't do better (adjacent words)
            break
        if positions_a[i] < positions_b[j]:
            i += 1
        else:
            j += 1
    return min_g


# ---------------------------------------------------------------------------
# Main scoring entry point
# ---------------------------------------------------------------------------

def score_document(
    posting_map: dict[str, tuple],   # term → (doc_id, ttf, btf, t_pos, b_pos)
    term_weights: dict[str, float],  # term → weight (1.0 for orig, γ for expanded)
    doc_stats:   tuple[int, int],    # (title_len, body_len)
    coll_stats:  dict,
    original_terms: list[str],       # used for phrase/proximity (unweighted)
) -> float:
    """
    Compute the full score for one document.

    *posting_map* contains only the terms that actually appear in this document.
    *term_weights* covers all query terms (original + expanded).
    """
    N          = coll_stats["N"]
    avg_title  = coll_stats["avg_title_len"]
    avg_body   = coll_stats["avg_body_len"]
    title_len, body_len = doc_stats

    bm25f_score = 0.0

    # -----------------------------------------------------------------------
    # BM25F component
    # -----------------------------------------------------------------------
    for term, weight in term_weights.items():
        entry = posting_map.get(term)
        if entry is None:
            continue  # term not in this doc
        _doc_id, title_tf, body_tf, _t_pos, _b_pos = entry

        # df is fetched from the global inverted index by the caller (passed via posting_map)
        df = entry[5] if len(entry) > 5 else 0
        bm25f_score += _bm25f_term(
            title_tf, body_tf,
            title_len, body_len,
            avg_title, avg_body,
            df, N,
            term_weight=weight,
        )

    # -----------------------------------------------------------------------
    # Phrase bonus (consecutive original query terms)
    # -----------------------------------------------------------------------
    phrase_score = 0.0
    orig_in_doc  = [t for t in original_terms if t in posting_map]

    for i in range(len(orig_in_doc) - 1):
        t_a = orig_in_doc[i]
        t_b = orig_in_doc[i + 1]
        _, _, _, t_pos_a, b_pos_a = posting_map[t_a][:5]
        _, _, _, t_pos_b, b_pos_b = posting_map[t_b][:5]

        if (_exact_phrase_positions(t_pos_a, t_pos_b) or
                _exact_phrase_positions(b_pos_a, b_pos_b)):
            phrase_score += config.PHRASE_BONUS

    # -----------------------------------------------------------------------
    # Proximity bonus (unordered window, all pairs of original terms)
    # -----------------------------------------------------------------------
    prox_score = 0.0
    W = config.PROXIMITY_WINDOW

    for t_a, t_b in combinations(orig_in_doc, 2):
        _, _, _, t_pos_a, b_pos_a = posting_map[t_a][:5]
        _, _, _, t_pos_b, b_pos_b = posting_map[t_b][:5]

        # Check title positions
        gap_t = _min_gap(t_pos_a, t_pos_b)
        if gap_t <= W:
            prox_score += config.PROXIMITY_BONUS_MAX * (1.0 - gap_t / W)

        # Check body positions
        gap_b = _min_gap(b_pos_a, b_pos_b)
        if gap_b <= W:
            prox_score += config.PROXIMITY_BONUS_MAX * (1.0 - gap_b / W)

    return bm25f_score + phrase_score + prox_score


# ---------------------------------------------------------------------------
# Batch ranking: score all candidate documents for a query
# ---------------------------------------------------------------------------

def rank_documents(
    term_weights: dict[str, float],
    inverted_index: dict,
    doc_stats: list[tuple[int, int]],
    collection_stats: dict,
    top_k: int = 1000,
) -> list[tuple[float, int]]:
    """
    Retrieve and score all documents that contain at least one query term.

    Returns a list of (score, doc_id) sorted descending by score, capped at top_k.

    Algorithm
    ---------
    1. For each query term, look up its posting list.
    2. For each document in those posting lists, accumulate the BM25F term
       contribution *directly* (avoiding a full posting_map per document).
    3. Add phrase and proximity bonuses using stored position data.
    """
    N         = collection_stats["N"]
    avg_title = collection_stats["avg_title_len"]
    avg_body  = collection_stats["avg_body_len"]

    # doc_id → accumulated score
    scores: dict[int, float] = {}
    # doc_id → {term: posting_tuple}  (needed for phrase/proximity)
    doc_postings: dict[int, dict[str, tuple]] = {}

    # -----------------------------------------------------------------------
    # BM25F accumulation
    # -----------------------------------------------------------------------
    for term, weight in term_weights.items():
        entry = inverted_index.get(term)
        if entry is None:
            continue
        df, postings = entry

        for posting in postings:
            doc_id, title_tf, body_tf, t_pos, b_pos = posting
            title_len, body_len = doc_stats[doc_id]

            term_score = _bm25f_term(
                title_tf, body_tf,
                title_len, body_len,
                avg_title, avg_body,
                df, N,
                term_weight=weight,
            )
            scores[doc_id] = scores.get(doc_id, 0.0) + term_score

            # Store posting for phrase/proximity (use weight=1.0 marker as 6th element)
            if doc_id not in doc_postings:
                doc_postings[doc_id] = {}
            # Store (doc_id, ttf, btf, t_pos, b_pos, df) so score_document can read df
            doc_postings[doc_id][term] = (doc_id, title_tf, body_tf, t_pos, b_pos, df)

    if not scores:
        return []

    # -----------------------------------------------------------------------
    # Phrase + proximity bonus (only for documents that matched ≥2 orig terms)
    # -----------------------------------------------------------------------
    original_terms = [t for t, w in term_weights.items() if w >= 1.0]

    W = config.PROXIMITY_WINDOW
    for doc_id, p_map in doc_postings.items():
        orig_in_doc = [t for t in original_terms if t in p_map]
        if len(orig_in_doc) < 2:
            continue

        phrase_score = 0.0
        prox_score   = 0.0

        # Phrase bonus
        for i in range(len(orig_in_doc) - 1):
            t_a, t_b = orig_in_doc[i], orig_in_doc[i + 1]
            t_pos_a, b_pos_a = p_map[t_a][3], p_map[t_a][4]
            t_pos_b, b_pos_b = p_map[t_b][3], p_map[t_b][4]
            if (_exact_phrase_positions(t_pos_a, t_pos_b) or
                    _exact_phrase_positions(b_pos_a, b_pos_b)):
                phrase_score += config.PHRASE_BONUS

        # Proximity bonus (all pairs)
        for t_a, t_b in combinations(orig_in_doc, 2):
            t_pos_a, b_pos_a = p_map[t_a][3], p_map[t_a][4]
            t_pos_b, b_pos_b = p_map[t_b][3], p_map[t_b][4]

            gap_t = _min_gap(t_pos_a, t_pos_b)
            if gap_t <= W:
                prox_score += config.PROXIMITY_BONUS_MAX * (1.0 - gap_t / W)

            gap_b = _min_gap(b_pos_a, b_pos_b)
            if gap_b <= W:
                prox_score += config.PROXIMITY_BONUS_MAX * (1.0 - gap_b / W)

        scores[doc_id] += phrase_score + prox_score

    # -----------------------------------------------------------------------
    # Sort and return top-k
    # -----------------------------------------------------------------------
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [(score, doc_id) for doc_id, score in ranked[:top_k]]


In [ ]:
import importlib
import config, preprocess, parse_docs, index_store, build_index, query_expand, rank
for mod in (config, preprocess, parse_docs, index_store, build_index, query_expand, rank):
    importlib.reload(mod)
print('All modules OK.')
print('TREC Disk 4:', config.DISK4)
print('TREC Disk 5:', config.DISK5)

---
## Section 3 — Build index
**Skip if `index_data/` already exists on Drive.**

Expected time: ~30–60 min on a free Colab CPU instance.
Checkpoints are saved to Drive every 20,000 docs — safe to resume after disconnection.

In [ ]:
import os, config
if os.path.exists(config.INDEX_FILE):
    print('Index already built:', config.INDEX_FILE)
    print('Skip to Section 4.')
else:
    print('No index found. Run the next cell to build.')

In [ ]:
import importlib, os, build_index, config
importlib.reload(config)
importlib.reload(build_index)

if not os.path.exists(config.INDEX_FILE):
    build_index.build()
else:
    print('Already built — skipping.')

In [ ]:
import os, config
print('Index files:')
for fname in sorted(os.listdir(config.INDEX_DIR)):
    fpath = os.path.join(config.INDEX_DIR, fname)
    if os.path.isfile(fpath):
        print(f'  {fname:<42s}  {os.path.getsize(fpath)/1e6:6.1f} MB')

---
## Section 4 — Load index
Run once per session (takes a few seconds).

In [ ]:
import importlib, os, time, pickle
import config, index_store, preprocess, query_expand, rank as ranking
for mod in (config, index_store, preprocess, query_expand, ranking):
    importlib.reload(mod)

def _pkl(p):
    with open(p, 'rb') as f: return pickle.load(f)

t0 = time.time()
inverted_index   = index_store.SQLiteIndex(config.INDEX_FILE)
doc_map          = _pkl(config.DOC_MAP_FILE)
doc_stats        = _pkl(config.DOC_STATS_FILE)
collection_stats = _pkl(config.COLL_STATS_FILE)
snippets = _pkl(config.SNIPPETS_FILE) if os.path.exists(config.SNIPPETS_FILE) else None

print(f'Loaded in {time.time()-t0:.1f}s')
print(f"Documents : {collection_stats['N']:,}")
print(f"Avg body  : {collection_stats['avg_body_len']:.0f} tokens")

---
## Section 5 — Search
Type a query and press **Enter** or click **Search**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import time

query_box = widgets.Text(
    placeholder='e.g. child support enforcement',
    description='Query:',
    layout=widgets.Layout(width='560px'),
)
topk_slider = widgets.IntSlider(
    value=10, min=5, max=100, step=5,
    description='Results:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px'),
)
expand_toggle = widgets.Checkbox(value=True,  description='WordNet expansion')
debug_toggle  = widgets.Checkbox(value=False, description='Show expanded terms')
search_btn    = widgets.Button(description='Search', button_style='primary',
                               layout=widgets.Layout(width='100px'))
output_area   = widgets.Output()

display(widgets.VBox([
    widgets.HBox([query_box, search_btn]),
    widgets.HBox([topk_slider, expand_toggle, debug_toggle]),
]), output_area)

def run_search(_):
    query_text = query_box.value.strip()
    if not query_text:
        return
    with output_area:
        clear_output(wait=True)
        t0 = time.time()
        norm = preprocess.normalise(query_text)
        if not norm:
            print('No searchable terms.')
            return
        surface = [s for s, _, _ in norm]
        terms   = list(dict.fromkeys(st for _, st, _ in norm))
        weights = (
            query_expand.expand_query(terms, surface, inverted_index, collection_stats)
            if expand_toggle.value else {t: 1.0 for t in terms}
        )
        ranked = ranking.rank_documents(
            weights, inverted_index, doc_stats, collection_stats,
            top_k=topk_slider.value,
        )
        ms = (time.time() - t0) * 1000
        if debug_toggle.value and expand_toggle.value:
            orig  = ', '.join(t for t, w in weights.items() if w >= 1.0)
            expnd = ', '.join(f'{t}({w:.2f})' for t, w in weights.items() if w < 1.0)
            display(HTML(f'<b>Original:</b> {orig}'))
            if expnd:
                display(HTML(f'<b>Expanded:</b> {expnd}'))
            display(HTML('<hr>'))
        if not ranked:
            print('No results.')
            return
        display(HTML(f'<p><b>{len(ranked)} results</b> &nbsp;&#183;&nbsp; {ms:.0f} ms &nbsp;&#183;&nbsp; <i>{query_text}</i></p>'))
        rows = ''
        for i, (score, doc_id) in enumerate(ranked, 1):
            docno = doc_map[doc_id]
            snip  = ''
            if snippets and doc_id < len(snippets):
                snip = snippets[doc_id]
                if len(snip) == config.SNIPPET_LENGTH:
                    snip += '...'
            rows += (
                f'<tr style="border-bottom:1px solid #eee">'
                f'<td style="padding:5px 10px;color:#aaa">{i}</td>'
                f'<td style="padding:5px 10px"><b>{docno}</b>'
                + (f'<br><small style="color:#555">{snip}</small>' if snip else '')
                + '</td>'
                f'<td style="padding:5px 10px;text-align:right;font-family:monospace">'
                f'{score:.4f}</td></tr>'
            )
        display(HTML(
            '<table style="width:100%;border-collapse:collapse">'
            '<thead><tr style="border-bottom:2px solid #ccc">'
            '<th style="padding:5px 10px;text-align:left">#</th>'
            '<th style="padding:5px 10px;text-align:left">Document</th>'
            '<th style="padding:5px 10px;text-align:right">Score</th>'
            f'</tr></thead><tbody>{rows}</tbody></table>'
        ))

search_btn.on_click(run_search)
query_box.on_submit(run_search)
